# Backfill: Umami export → monthly rollups

Umami is gone and the `analytics` counter starts from zero, so ~7 months of
history only exists in the cloud.umami.is data export. This notebook folds that
export into the **monthly rollup** layout the readout reads:

```
rollup/{site}/{YYYY-MM}.json
```

It never touches the hourly `counts/` prefix the live counter writes, and it is
idempotent — a day already present in a stored rollup is never overwritten.

**Why monthly and not hourly.** `listObjects`
(`shared/s3-utils/src/index.ts`) issues a single un-paginated ListObjectsV2 —
max 1000 keys, silently truncated — and hourly objects accrue at 8760/year.
Replaying this export as hourly objects would add 874 keys and put the
`counts/fretchen.eu/` prefix within ~10% of that ceiling on day one. Rollup keys
are *computed* from a date range instead of listed, so there is no ceiling and a
month of traffic costs one GET.

**Privacy.** `website_event.csv` carries `session_id`, city, region, country,
device, screen and language. `umami_backfill.py` projects to (day, path) → count
and drops everything else, so the rollups hold no more than the live counter
would have recorded. Delete the export when you're done — `analytics/.gitignore`
covers `*.zip`/`*.csv` so it can't be committed by accident.


In [1]:
import re
import zipfile
from pathlib import Path

import os
from dotenv import load_dotenv

from storage import LocalStorage, S3Storage
from umami_backfill import (
    SITE,
    backfill,
    normalize_path,
    read_pageviews,
    to_monthly_rollups,
)

load_dotenv()  # searches upward — finds ../.env (analytics/.env)


True

## 0. Unpack the export

The zip is whatever cloud.umami.is named it (a website-id UUID). Only
`website_event.csv` is used: `event_data.csv` is custom-event payloads and
`session_data.csv` is empty.


In [2]:
EXPORT_ZIP = Path("../95618d92-18ca-46f4-8a9e-b0556f54bab8.zip")
EXPORT_DIR = Path("export")

with zipfile.ZipFile(EXPORT_ZIP) as zf:
    zf.extractall(EXPORT_DIR)

CSV_PATH = EXPORT_DIR / "website_event.csv"
print(sorted(p.name for p in EXPORT_DIR.iterdir()))


['event_data.csv', 'session_data.csv', 'website_event.csv']


## 1. Normalisation — the part that decides whether old and new data line up

Umami logged the raw browser pathname. The beacon reports
`pageContext.urlPathname`, which Vike derives from `urlLogical` — set by
`website/pages/+onBeforeRoute.ts` via `extractLocale()`. That means the live
counter records the **canonical sitemap form**:

| Rule | Umami logged | Beacon records |
| --- | --- | --- |
| trailing slash forced on non-root paths | `/amo/11` | `/amo/11/` |
| locale prefix stripped | `/de/blog/25/` | `/blog/25/` |
| fragment dropped | `/blog/22/#eine-interpretation` | `/blog/22/` |
| query dropped | `/x402/?ref=x` | `/x402/` |

`generateSitemap.ts` applies the identical rule (`getLocaleInfo` strips the
locale, `filePathToUrlPath` forces the trailing slash), so normalising the
import to sitemap form makes the backfilled and live series directly
comparable — and both line up with `sitemap.xml`.

**Consequence worth knowing:** German pages are *not* distinguishable. `/de/`
traffic folds into the English path both historically and going forward,
because the beacon never sees the locale. Fixing that means sending the locale
as a separate field — deliberately out of scope for now.


In [3]:
checks = {
    "/": "/",
    "/blog/25/": "/blog/25/",
    "/amo/11": "/amo/11/",
    "/de/": "/",
    "/de/blog/25/": "/blog/25/",
    "/blog/22/#user-content-fnref-1": "/blog/22/",
    "/x402/?ref=x": "/x402/",
    "not-a-path": None,
}

for raw, expected in checks.items():
    got = normalize_path(raw)
    assert got == expected, f"{raw!r} -> {got!r}, expected {expected!r}"
print(f"{len(checks)} normalisation cases OK")


8 normalisation cases OK


## 2. Dry run — aggregate, don't write anywhere real

`LocalStorage()` writes to `state/`, the same directory `npm run dev`'s
`FileHitStorage` uses. Safe to re-run.


In [4]:
pageviews = read_pageviews(CSV_PATH)
rollups = to_monthly_rollups(pageviews)

print(f"{len(pageviews)} pageviews -> {len(rollups)} monthly objects\n")
for month, rollup in sorted(rollups.items()):
    hits = sum(day["hits"] for day in rollup["days"].values())
    print(f"  {month}  {len(rollup['days']):>2} days  {hits:>5} hits")


1612 pageviews -> 8 monthly objects

  2026-01  29 days    199 hits
  2026-02  28 days    184 hits
  2026-03  31 days    315 hits
  2026-04  29 days    202 hits
  2026-05  30 days    217 hits
  2026-06  29 days    203 hits
  2026-07  28 days    155 hits
  2026-08  10 days    137 hits


In [5]:
dry = LocalStorage()
report = backfill(dry, CSV_PATH)

for month, entry in report.items():
    skipped = f"  (skipped {len(entry['days_skipped'])} existing)" if entry["days_skipped"] else ""
    print(f"{entry['key']}  {entry['days_written']} days, {entry['hits']} hits{skipped}")


rollup/fretchen.eu/2026-01.json  0 days, 199 hits  (skipped 29 existing)
rollup/fretchen.eu/2026-02.json  0 days, 184 hits  (skipped 28 existing)
rollup/fretchen.eu/2026-03.json  0 days, 315 hits  (skipped 31 existing)
rollup/fretchen.eu/2026-04.json  0 days, 202 hits  (skipped 29 existing)
rollup/fretchen.eu/2026-05.json  0 days, 217 hits  (skipped 30 existing)
rollup/fretchen.eu/2026-06.json  0 days, 203 hits  (skipped 29 existing)
rollup/fretchen.eu/2026-07.json  0 days, 155 hits  (skipped 28 existing)
rollup/fretchen.eu/2026-08.json  0 days, 137 hits  (skipped 10 existing)


## 3. Sanity-check the paths against `sitemap.xml`

Anything not in the sitemap is either a page that has since moved or a real
404 someone hit. Both are legitimate history — this is a look-at-it check, not
a filter.


In [6]:
SITEMAP = Path("../../website/build/sitemap.xml")  # produced by `npm run build`

sitemap_paths = set(re.findall(r"<loc>https://www\.fretchen\.eu(.*?)</loc>", SITEMAP.read_text()))
counts = {}
for _, path in pageviews:
    counts[path] = counts.get(path, 0) + 1

unknown = sorted(set(counts) - sitemap_paths, key=lambda p: -counts[p])
print(f"{len(counts)} canonical paths, {len(sitemap_paths)} in sitemap, {len(unknown)} unknown\n")
for path in unknown[:20]:
    print(f"{counts[path]:>5}  {path}")


114 canonical paths, 81 in sitemap, 35 unknown

   30  /amo/11/
   26  /amo/13/
   24  /amo/10/
    7  /amo/8/
    6  /amo/12/
    6  /amo/5/
    3  /amo/16/
    3  /amo/2/
    3  /amo/18/
    2  /blog/11/7/
    2  /daniel-website/
    2  /AutoIncentive/
    2  /blog/quantum/
    2  /quantum/qml/1/qml102/
    1  /quantum/hardware/qhw2/
    1  /amo/6/
    1  /blog/9/6/
    1  /quantum/qml/2/1/
    1  /quantum/qml/qml101/
    1  /quantum/qml/3/2/


## 4. The real write

Flip the flag. Eight new objects under a prefix nothing else writes to;
re-running is a no-op because existing days always win.


In [7]:
WRITE_TO_S3 = True  # flip to True to write for real

if WRITE_TO_S3:
    s3 = S3Storage(
        access_key=os.environ["SCW_ACCESS_KEY"],
        secret_key=os.environ["SCW_SECRET_KEY"],
    )
    report = backfill(s3, CSV_PATH)
    for month, entry in report.items():
        print(f"{entry['key']}  {entry['days_written']} days, {entry['hits']} hits")
else:
    print("dry run only — nothing written to S3")


rollup/fretchen.eu/2026-01.json  29 days, 199 hits
rollup/fretchen.eu/2026-02.json  28 days, 184 hits
rollup/fretchen.eu/2026-03.json  31 days, 315 hits
rollup/fretchen.eu/2026-04.json  29 days, 202 hits
rollup/fretchen.eu/2026-05.json  30 days, 217 hits
rollup/fretchen.eu/2026-06.json  29 days, 203 hits
rollup/fretchen.eu/2026-07.json  28 days, 155 hits
rollup/fretchen.eu/2026-08.json  10 days, 137 hits


## 5. Read it back

Confirms the objects landed and the totals survived the round trip. The full
range readout lives in `02_readout.ipynb`.


In [8]:
if WRITE_TO_S3:
    for month in sorted(rollups):
        stored = s3.read(f"rollup/{SITE}/{month}.json")
        hits = sum(day["hits"] for day in stored["days"].values())
        print(f"{month}: {len(stored['days'])} days, {hits} hits")


2026-01: 29 days, 199 hits
2026-02: 28 days, 184 hits
2026-03: 31 days, 315 hits
2026-04: 29 days, 202 hits
2026-05: 30 days, 217 hits
2026-06: 29 days, 203 hits
2026-07: 28 days, 155 hits
2026-08: 10 days, 137 hits
